### 1. Install & Import Library

In [ ]:
import subprocess, sys
def _ensure_pkg(import_name, pip_spec=None):
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_spec or import_name])
_ensure_pkg('monai')
_ensure_pkg('nibabel')
_ensure_pkg('sklearn', 'scikit-learn')
_ensure_pkg('gdown')
import os, re, glob, zipfile, random, time, json, warnings, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import nibabel as nib
from scipy import ndimage

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, classification_report)

import monai
from monai.networks.nets import DenseNet121

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Monai version:', monai.__version__)
NOTEBOOK_REV = 6  
print('Notebook REV:', NOTEBOOK_REV)


def set_seed(seed):
    """Set semua random seed utk reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


### 2. Konfigurasi

Konfigurasi mode (Kaggle vs lokal), parameter training, dan path data.
Folder nama dengan spasi dan titik ditangani dengan glob + quoting.


In [ ]:
# ============================================================
#  KONFIGURASI — 3D Cardio Classification (Kaggle / Colab / lokal)
# ============================================================
IS_KAGGLE = ('/kaggle' in __import__('os').getcwd()) or __import__('os').path.exists('/kaggle')
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_KAGGLE:
    MODE = 'kaggle'
elif IS_COLAB:
    MODE = 'colab'
else:
    MODE = 'local'

GDRIVE_FILE_ID = '1q2aRIgmmv5IdqlTFQUbYTV9hVzOzvC4x'
RAR_FILENAME   = 'data.rar'

if IS_KAGGLE:
    RAR_PATH   = '/kaggle/working/data.rar'
    RAR_DIR    = '/kaggle/working/data'
    OUTPUT_DIR = '/kaggle/working/outputs_3d'
    NUM_WORKERS = 2
elif IS_COLAB:
    RAR_PATH   = '/content/data.rar'
    RAR_DIR    = '/content/data'
    OUTPUT_DIR = '/content/outputs_3d'
    NUM_WORKERS = 2
else:
    RAR_PATH   = RAR_FILENAME
    RAR_DIR    = 'data'
    OUTPUT_DIR = 'outputs_3d'
    NUM_WORKERS = 2

BASE_DIR = '3D_CARDIO'
if MODE == 'local':
    _train_dirs = sorted(glob.glob(os.path.join(BASE_DIR, '1.*', 'CARDIO')))
    _test_dirs  = sorted(glob.glob(os.path.join(BASE_DIR, '2.*', 'CARDIO')))
    assert _train_dirs, 'Folder training CARDIO tidak ditemukan!'
    assert _test_dirs,  'Folder testing CARDIO tidak ditemukan!'
    TRAIN_CARDIO_DIR = _train_dirs[0]
    TEST_CARDIO_DIR  = _test_dirs[0]
else:
    TRAIN_CARDIO_DIR, TEST_CARDIO_DIR = None, None

# ===== TRAINING PARAMETERS =====
IMG_SIZE   = 128
IMG_DEPTH  = 48
BATCH_SIZE = 4
EPOCHS     = 40
LR         = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE   = 12
SEEDS      = [42, 43, 44]

# ===== MODEL =====
INIT_FEATURES = 64
GROWTH_RATE   = 32

# ===== LOSS =====
FOCAL_GAMMA   = 2.0
LABEL_SMOOTH  = 0.1

# ===== TRAINING =====
AMP          = True
GRAD_CLIP    = 1.0
SWA          = True
SWA_EPOCHS   = 3
EMA_ALPHA    = 0.5
WARMUP_EPOCHS = 1

# ===== EVAL =====
TUNE_STEP    = 0.02
TUNE_CLAMP   = (0.30, 0.70)
TUNE_MIN_GAIN = 0.02
TUNE_TOLERATE = 0.999 
TUNE_MAX_POSRATE = 0.85
BOOTSTRAP_N  = 2000

# ===== DATA =====
N_WORKERS    = NUM_WORKERS
PIN_MEMORY   = True

os.makedirs(OUTPUT_DIR, exist_ok=True)

CONFIG = dict(mode=MODE, gdrive_file_id=GDRIVE_FILE_ID, img_size=IMG_SIZE,
              img_depth=IMG_DEPTH, batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR,
              weight_decay=WEIGHT_DECAY, patience=PATIENCE, seeds=SEEDS,
              init_features=INIT_FEATURES, growth_rate=GROWTH_RATE,
              focal_gamma=FOCAL_GAMMA, label_smooth=LABEL_SMOOTH,
              amp=AMP, grad_clip=GRAD_CLIP, swa=SWA, swa_epochs=SWA_EPOCHS,
              ema_alpha=EMA_ALPHA, warmup_epochs=WARMUP_EPOCHS,
              tune_step=TUNE_STEP, tune_clamp=TUNE_CLAMP,
              tune_min_gain=TUNE_MIN_GAIN, bootstrap_n=BOOTSTRAP_N,
              num_workers=NUM_WORKERS, output_dir=OUTPUT_DIR)

print('IS_KAGGLE:', IS_KAGGLE, '| IS_COLAB:', IS_COLAB, '| Mode:', MODE, '| Device:', device)
print('TRAIN_CARDIO_DIR:', TRAIN_CARDIO_DIR)
print('TEST_CARDIO_DIR:', TEST_CARDIO_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Epochs:', EPOCHS, '| Batch:', BATCH_SIZE, '| IMG:', IMG_SIZE, 'x', IMG_SIZE, 'x', IMG_DEPTH)
print('Seeds:', SEEDS)


### 3. Download Dataset RAR dari Google Drive & Ekstrak

Download `data.rar` via `gdown` (API Python),
lalu ekstrak dengan `unrar` — Kaggle ke `/kaggle/working/data/`, Colab ke `/content/data/`.
Fallback: dataset yang sudah di-attach (`/kaggle/input`) atau RAR yang sudah ada (resume).
Mode lokal: pakai folder `3D_CARDIO/`.


In [ ]:
# ============================================================
#  DOWNLOAD & EXTRACT — 3-mode: kaggle | colab (gdown) | local
#  Mengembalikan DATA ROOT (folder berisi '1. Training...' & '2. Testing...').
# ============================================================
def _apt_install_unrar(tag):
    """Install binary unrar. Colab jalan sbg root (tanpa sudo), Kaggle perlu sudo."""
    ip = get_ipython()
    ip.system('apt-get update -qq 2>/dev/null && apt-get install -qq -y unrar 2>/dev/null || sudo apt-get update -qq 2>/dev/null && sudo apt-get install -qq -y unrar 2>/dev/null || true')
    print(f'[{tag}] unrar siap (atau sudah ada).')


def _gdown_download(file_id, output, tag):
    """Download file besar dari Google Drive (bypass virus-scan confirm)."""
    import gdown
    print(f'[{tag}] Download {output} (file ID: {file_id}) ...')
    gdown.download(id=file_id, output=output, quiet=False)
    size = os.path.getsize(output)
    print(f'[{tag}] size: {size / 1e6:.1f} MB')
    assert size > 1e6, f'{output} terlalu kecil ({size} byte) — download gagal!'
    return output


def _find_data_root(search_dir, tag):
    """Cari folder berisi '1.*' (training) — tahan terhadap nested extract."""
    cands = sorted(glob.glob(os.path.join(search_dir, '**', '1.*'), recursive=True))
    cands = [c for c in cands if os.path.isdir(c)]
    assert cands, f'[{tag}] folder training (1.*) tidak ditemukan di {search_dir}!'
    root = os.path.dirname(cands[0])
    print(f'[{tag}] data root: {root}')
    print(f'[{tag}] isi root: {sorted(os.listdir(root))}')
    return root


def download_and_extract():
    """Download data.rar (bila perlu), extract, return DATA ROOT."""
    if MODE in ('kaggle', 'colab'):
        tag = MODE.upper()
        # 1) Dataset sudah di-attach? (Kaggle Datasets / Drive mount)
        pre = sorted(glob.glob('/kaggle/input/**/1.*', recursive=True)) if MODE == 'kaggle' else []
        pre = [p for p in pre if os.path.isdir(p)]
        if pre:
            root = os.path.dirname(pre[0])
            print(f'[{tag}] dataset sudah ada: {root}')
            return root
        # 2) RAR sudah ada (resume) atau download
        if os.path.exists(RAR_PATH):
            print(f'[{tag}] {RAR_PATH} sudah ada, skip download')
        else:
            _gdown_download(GDRIVE_FILE_ID, RAR_PATH, tag)
        # 3) Extract (skip bila sudah ada dari run sebelumnya)
        already = sorted(glob.glob(os.path.join(RAR_DIR, '**', '1.*'), recursive=True))
        if not already:
            _apt_install_unrar(tag)
            os.makedirs(RAR_DIR, exist_ok=True)
            print(f'[{tag}] Extract {RAR_PATH} -> {RAR_DIR} ...')
            get_ipython().system(f'unrar x -o+ "{RAR_PATH}" "{RAR_DIR}/"')
        return _find_data_root(RAR_DIR, tag)
    else:
        # Local mode: pakai folder 3D_CARDIO/
        assert os.path.isdir(BASE_DIR), f'Folder {BASE_DIR} tidak ditemukan (mode local).'
        print(f'[LOCAL] Mode lokal → pakai folder: {BASE_DIR}')
        print(f'[LOCAL] Isi:', sorted(os.listdir(BASE_DIR)))
        return BASE_DIR

DATA_ROOT = download_and_extract()
BASE_DIR = DATA_ROOT  # resolve_dataset memakai BASE_DIR
_train_dirs = sorted(glob.glob(os.path.join(BASE_DIR, '1.*', 'CARDIO')))
_test_dirs  = sorted(glob.glob(os.path.join(BASE_DIR, '2.*', 'CARDIO')))
assert _train_dirs and _test_dirs, f'CARDIO tidak ditemukan di {BASE_DIR}!'
TRAIN_CARDIO_DIR, TEST_CARDIO_DIR = _train_dirs[0], _test_dirs[0]
print('DATA_ROOT:', DATA_ROOT)


### 4. Resolve Dataset — Kumpulkan File NIfTI + Label



In [ ]:
# ============================================================
#  RESOLVE DATASET — kumpulkan file NIfTI + label
# ============================================================
def resolve_dataset(data_root):
    """Browse folder CARDIO/NON_CARDIO di training dan testing.
    
    Return:
        train_files: list of (path, label)
        test_files: list of (path, label)
        subject_ids: set of all subject IDs (nama file tanpa .nii)
    """
    # Gunakan glob untuk handle folder dengan spasi + titik
    # TODO
    
    
    train_files, test_files = [], []
    subject_ids = set()
    
    # Training: CARDIO=1, NON_CARDIO=0
    for d in train_dirs:
        for f in sorted(os.listdir(d)):
            if f.endswith('.nii'):
                path = os.path.join(d, f)
                sid = f.replace('.nii', '')
                train_files.append((path, 1, sid))
                subject_ids.add(sid)
    for d in train_non_dirs:
        for f in sorted(os.listdir(d)):
            if f.endswith('.nii'):
                path = os.path.join(d, f)
                sid = f.replace('.nii', '')
                train_files.append((path, 0, sid))
                subject_ids.add(sid)
    
    # Testing: CARDIO=1, NON_CARDIO=0
    for d in test_dirs:
        for f in sorted(os.listdir(d)):
            if f.endswith('.nii'):
                path = os.path.join(d, f)
                sid = f.replace('.nii', '')
                test_files.append((path, 1, sid))
                subject_ids.add(sid)
    for d in test_non_dirs:
        for f in sorted(os.listdir(d)):
            if f.endswith('.nii'):
                path = os.path.join(d, f)
                sid = f.replace('.nii', '')
                test_files.append((path, 0, sid))
                subject_ids.add(sid)
    
    return train_files, test_files, subject_ids

train_files, test_files, all_subjects = resolve_dataset(DATA_ROOT)
print(f'Train files: {len(train_files)} | Test files: {len(test_files)} | Total subjects: {len(all_subjects)}')

# Verifikasi distribusi kelas
train_cardio = sum(1 for _, l, _ in train_files if l == 1)
train_non = sum(1 for _, l, _ in train_files if l == 0)
test_cardio = sum(1 for _, l, _ in test_files if l == 1)
test_non = sum(1 for _, l, _ in test_files if l == 0)
print(f'Train: CARDIO={train_cardio} NON_CARDIO={train_non}')
print(f'Test:  CARDIO={test_cardio} NON_CARDIO={test_non}')

# Verifikasi tidak ada overlap subject ID
train_ids = {sid for _, _, sid in train_files}
test_ids = {sid for _, _, sid in test_files}
overlap = train_ids & test_ids
assert len(overlap) == 0, f'OVERLAP subject ID antar train/test: {overlap}'
print('✔ Tidak ada overlap subject ID antar train/test split.')


### 5. Exploratory Data Analysis (EDA)

Analisis shape, dtype, nilai min/max/mean per volume NIfTI.
Visualisasi distribusi shape dan mean intensity.


In [ ]:
# ============================================================
#  EDA — Analisis NIfTI files
# ============================================================
def load_nifti_info(path):
    """Load NIfTI dan return shape, dtype, stats."""
    img = nib.load(path)
    data = img.get_fdata()
    return {
        'path': path,
        'shape': data.shape,
        'dtype': str(data.dtype),
        'min': float(data.min()),
        'max': float(data.max()),
        'mean': float(data.mean()),
        'std': float(data.std()),
    }

# Sample semua file untuk EDA
print('=== SAMPLE NIfTI INFO (5 file) ===')
for f_info in train_files[:5]:
    info = load_nifti_info(f_info[0])
    print(f"  {os.path.basename(info['path'])}: shape={info['shape']}, "
          f"dtype={info['dtype']}, mean={info['mean']:.1f}")

# Statistik lengkap
print(f'\n=== STATISTIK LENGKAP ===')
all_infos = [load_nifti_info(p) for p, _, _ in train_files]
shapes = [i['shape'] for i in all_infos]
means = [i['mean'] for i in all_infos]
stds = [i['std'] for i in all_infos]
dtypes = set(i['dtype'] for i in all_infos)

print(f'Total train volumes: {len(all_infos)}')
print(f'Dtypes: {dtypes}')
print(f'Shape unik: {sorted(set(shapes))}')
print(f'Mean intensity: min={min(means):.1f} max={max(means):.1f} mean={np.mean(means):.1f}')
print(f'Std intensity: min={min(stds):.1f} max={max(stds):.1f}')

# Distribusi shape
shape_strs = ['x'.join(map(str, s)) for s in shapes]
shape_counts = pd.Series(shape_strs).value_counts()
print(f'\nDistribusi shape:')
for s, c in shape_counts.items():
    print(f'  {s}: {c} volumes')

# Visualisasi
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist([i['mean'] for i in all_infos if i['path'] and 'CARDIO' in i['path']],
             bins=20, alpha=0.7, label='CARDIO', color='#e74c3c')
axes[0].hist([i['mean'] for i in all_infos if i['path'] and 'NON_CARDIO' in i['path']],
             bins=20, alpha=0.7, label='NON_CARDIO', color='#3498db')
axes[0].set_xlabel('Mean Intensity')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribusi Mean Intensity')
axes[0].legend()

axes[1].hist(shape_counts.values, bins=range(len(shape_counts)+1), alpha=0.7, color='#2ecc71')
axes[1].set_xlabel('Shape category')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribusi Shape Volume')

axes[2].scatter([i['mean'] for i in all_infos],
                [i['std'] for i in all_infos], alpha=0.5, s=20)
axes[2].set_xlabel('Mean')
axes[2].set_ylabel('Std')
axes[2].set_title('Mean vs Std')
plt.tight_layout()
plt.show()

print(f'\nVerifikasi data:')
print(f'  - Total file .nii lokal: {len(train_files) + len(test_files)}')
print(f'  - Shape bervariasi: {sorted(set(shapes))}')
print(f'  - dtype: {dtypes.pop()} (header uint16, get_fdata float64)')
print(f'  - Nilai: 0-4095 (uint16 range)')
print(f'  - 1 file = 1 pasien')
print(f'  - Label dari nama folder CARDIO/NON_CARDIO')


### 6. DataFrame & Split

Buat DataFrame dari file list, bersihkan subject ganda (`_dedupe_subjects`),
lalu split train menjadi train/val (85/15) dengan `train_test_split`
`stratify` seed 42. Verifikasi tidak ada overlap subject ID antar split.


In [ ]:
# Buat DataFrame dari file list
train_rows = [(p, l, sid) for p, l, sid in train_files]
test_rows  = [(p, l, sid) for p, l, sid in test_files]

train_df = pd.DataFrame(train_rows, columns=['path', 'label', 'subject'])
test_df  = pd.DataFrame(test_rows, columns=['path', 'label', 'subject'])

def _dedupe_subjects(df, name):
    """Bersihkan subject ID ganda (data full 771 pasien punya 1 duplikat).
    - Subject dgn label KONFLIK (ada di CARDIO & NON_CARDIO) -> buang semua barisnya.
    - Duplikat label sama -> keep first.
    """
    dup_mask = df.duplicated(subset='subject', keep=False)
    if dup_mask.any():
        dups = df[dup_mask].sort_values('subject')
        print(f'[!] {name}: {dups["subject"].nunique()} subject ganda ({len(dups)} baris):')
        for sid, g in dups.groupby('subject'):
            print(f'    subject={sid} label={sorted(g["label"].unique().tolist())}')
            for p in g['path'].tolist():
                print(f'      - {p}')
        conflict = dups.groupby('subject')['label'].nunique()
        conflict_ids = set(conflict[conflict > 1].index)
        if conflict_ids:
            print(f'[!] {name}: buang {len(conflict_ids)} subject konflik-label: {sorted(conflict_ids)}')
            df = df[~df['subject'].isin(conflict_ids)]
        df = df.drop_duplicates(subset='subject', keep='first')
    return df.reset_index(drop=True)


train_df = _dedupe_subjects(train_df, 'train')
test_df = _dedupe_subjects(test_df, 'test')

print(f'Train DataFrame: {len(train_df)} samples')
print(f'Test DataFrame:  {len(test_df)} samples')
print(f'Train class dist:\n{train_df["label"].value_counts().sort_index()}')
print(f'Test class dist:\n{test_df["label"].value_counts().sort_index()}')

# Stratified 85/15 split dari train (seed 42)
train_sub, val_sub = train_test_split(
    train_df, test_size=0.15, stratify=train_df['label'], random_state=42
)
print(f'\nSplit train/val: {len(train_sub)} / {len(val_sub)}')
print(f'Val class dist:\n{val_sub["label"].value_counts().sort_index()}')

# Verifikasi tidak ada overlap subject ID
train_ids = set(train_sub['subject'].values)
val_ids = set(val_sub['subject'].values)
test_ids = set(test_df['subject'].values)
assert train_ids.isdisjoint(val_ids), 'OVERLAP train-val!'
assert train_ids.isdisjoint(test_ids), 'OVERLAP train-test!'
assert val_ids.isdisjoint(test_ids), 'OVERLAP val-test!'
print('✔ Tidak ada overlap subject ID antar split (train/val/test).')

# Verifikasi subject ID unik per pasien
for name, df in [('train', train_sub), ('val', val_sub), ('test', test_df)]:
    n_files = len(df)
    n_subjects = df['subject'].nunique()
    print(f'  {name}: {n_files} files, {n_subjects} subjects (1 file = 1 pasien)')
    assert n_files == n_subjects, f'{name}: jumlah file != jumlah subject!'


### 7. Dataset MONAI 3D

Loader NIfTI 3D: `nibabel.get_fdata()` + resize ke `(128,128,48)` via `scipy.ndimage.zoom`,
clip `[0,4095]`, z-score per-volume (mean/std volume sendiri).
Augmentasi train: flip acak 3 aksis + rotasi 90 + Gaussian noise kecil.
Eval: tanpa augmentasi.


In [ ]:
# ============================================================
#  MONAI 3D DATASET — NIfTI loader + resize + z-score
# ============================================================
def _crop_or_pad_3d(vol, target):
    """Paksa shape eksak (H, W, D): center-crop bila kebesaran, zero-pad bila kekecilan.
    Pengaman karena resize/augmentasi bisa off-by-one utk shape tak terlihat
    di sample (full 771 pasien) -> torch.stack di DataLoader gagal bila batch campur shape.
    """
    slabs = []
    for s, t in zip(vol.shape, target):
        if s > t:
            st = (s - t) // 2
            slabs.append(slice(st, st + t))
        else:
            slabs.append(slice(None))
    vol = vol[tuple(slabs)]
    pad = []
    for s, t in zip(vol.shape, target):
        if s < t:
            b = (t - s) // 2
            pad.append((b, t - s - b))
        else:
            pad.append((0, 0))
    if any(p != (0, 0) for p in pad):
        vol = np.pad(vol, pad, mode='constant', constant_values=0)
    return vol


class NCCT3DDataset(Dataset):
    """Dataset 3D NIfTI untuk klasifikasi cardio vs non-cardio.
    
    - Load via nibabel.get_fdata()
    - Resize ke (IMG_SIZE, IMG_SIZE, IMG_DEPTH) via scipy.ndimage.zoom
    - Clip [0, 4095] lalu z-score per-volume (mean/std volume sendiri)
    - Augmentasi train: flip acak 3 aksis + rotasi 90 + Gaussian noise
    """
    def __init__(self, df, transform=None, mode='train'):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.mode = mode
        self.IMG_SIZE = IMG_SIZE
        self.IMG_DEPTH = IMG_DEPTH
    
    def __len__(self):
        return len(self.df)
    
    def _load_and_preprocess(self, path):
        """Load NIfTI, resize, clip, z-score per-volume."""
        img = nib.load(path)
        data = img.get_fdata().astype(np.float32)
        if data.ndim == 4:
            # NIfTI multi-volume (rekonstruksi ganda) -> pakai volume pertama
            data = data[..., 0].astype(np.float32)
        if data.ndim != 3:
            raise RuntimeError(f'File {path} berdimensi {data.ndim}, bukan volume 3D!')
        
        # Resize ke (IMG_SIZE, IMG_SIZE, IMG_DEPTH)
        if data.shape != (self.IMG_SIZE, self.IMG_SIZE, self.IMG_DEPTH):
            zoom_factors = (
                self.IMG_SIZE / data.shape[0],
                self.IMG_SIZE / data.shape[1],
                self.IMG_DEPTH / data.shape[2]
            )
            data = ndimage.zoom(data, zoom_factors, order=1)
        # Paksa shape eksak (pengaman off-by-one pembulatan utk shape baru)
        data = _crop_or_pad_3d(data, (self.IMG_SIZE, self.IMG_SIZE, self.IMG_DEPTH))
        
        # Clip [0, 4095]
        data = np.clip(data, 0, 4095)
        
        # Z-score per-volume (mean/std volume sendiri)
        vol_mean = data.mean()
        vol_std = data.std()
        if vol_std > 1e-6:
            data = (data - vol_mean) / vol_std
        
        return data  # (D, H, W)
    
    def _augment(self, data):
        """Augmentasi train: flip acak 3 aksis + rotasi 90 + Gaussian noise."""
        # Random flip 3 aksis
        for axis in range(3):
            if random.random() < 0.5:
                data = np.flip(data, axis=axis).copy()
        
        # Random rotation 90 derajat di bidang aksial (0,1) SAJA.
        # Plane (0,2)/(1,2) DILARANG: menukar sumbu depth (48) dgn spasial (128)
        # -> shape (48,128,128) tercampur dlm batch -> torch.stack gagal + anatomi rusak.
        k = random.randint(1, 3)
        data = np.rot90(data, k, axes=(0, 1))
        
        # Gaussian noise kecil
        noise = np.random.normal(0, 0.05, data.shape).astype(np.float32)
        data = data + noise
        
        return data
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row['path']
        label = int(row['label'])
        subject = row['subject']
        
        data = self._load_and_preprocess(path)
        
        if self.transform is not None and self.mode == 'train':
            data = self._augment(data)
        
        # Add channel dim: (1, H, W, D)
        data = np.expand_dims(data, 0).astype(np.float32)
        expected = (1, self.IMG_SIZE, self.IMG_SIZE, self.IMG_DEPTH)
        if tuple(data.shape) != expected:
            raise RuntimeError(f'SHAPE {tuple(data.shape)} != {expected} utk {path}')
        
        return torch.from_numpy(data), label, subject, path


def _train_transform(data):
    """Transform untuk train: augmentasi."""
    return data  # Augmentasi dilakukan di dalam __getitem__


def _eval_transform():
    """Eval: tanpa augmentasi, hanya preprocess."""
    return None


def scan_volume_shapes(df, name, max_report=10):
    """Pastikan semua file 3D/4D valid.
    Dijalankan SEBELUM training
    """
    bad, four_d = [], 0
    for _, row in df.iterrows():
        try:
            shp = nib.load(row['path']).shape
        except Exception as e:
            bad.append((row['path'], f'LOAD_FAIL: {e}'))
            continue
        if len(shp) == 4:
            four_d += 1  # ditangani _load_and_preprocess (pakai volume pertama)
        elif len(shp) != 3:
            bad.append((row['path'], f'ndim={len(shp)} shape={shp}'))
    print(f'[SCAN] {name}: {len(df)} file | 4D={four_d} | anomali={len(bad)}')
    for p, info in bad[:max_report]:
        print(f'    {info} <- {p}')
    assert not bad, f'{name}: ada {len(bad)} file anomali (lihat di atas)!'


def make_3d_loaders(train_df, val_df, test_df, batch_size=BATCH_SIZE):
    """Buat DataLoader untuk train/val/test."""
    scan_volume_shapes(train_df, 'train')
    scan_volume_shapes(val_df, 'val')
    scan_volume_shapes(test_df, 'test')
    
    # TODO
    
    return train_ld, val_ld, test_ld

print('3D Dataset classes siap.')
print(f'IMG_SIZE={IMG_SIZE}, IMG_DEPTH={IMG_DEPTH}, BATCH_SIZE={BATCH_SIZE}')


### 8. Model DenseNet121-3D

Model `monai.networks.nets.DenseNet121(spatial_dims=3, in_channels=1, out_channels=2,
init_features=64, growth_rate=32)`. Fallback ke simple 3D CNN jika MONAI tidak ada.


In [ ]:
# ============================================================
#  MODEL — DenseNet121-3D
# ============================================================
def build_model():
    try:
        # TODO
        print(f'[MODEL] DenseNet121-3D loaded (MONAI)')
        print(f'  init_features={INIT_FEATURES}, growth_rate={GROWTH_RATE}')
        return model
    except Exception as e:
        print(f'[MODEL] MONAI DenseNet121 gagal ({e}), fallback 3D CNN')
        return build_fallback_cnn()


def build_fallback_cnn():
    """Fallback simple 3D CNN (tanpa timm 2D)."""
    class Simple3DCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv3d(1, 32, 3, padding=1), nn.BatchNorm3d(32), nn.ReLU(),
                nn.Conv3d(32, 64, 3, padding=1), nn.BatchNorm3d(64), nn.ReLU(),
                nn.Conv3d(64, 128, 3, padding=1), nn.BatchNorm3d(128), nn.ReLU(),
                nn.AdaptiveAvgPool3d(1),
            )
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.4),
                nn.Linear(64, 2),
            )
        def forward(self, x):
            return self.classifier(self.features(x))
    model = Simple3DCNN()
    print(f'[MODEL] Fallback 3D CNN dibangun')
    return model

model = build_model()
model = model.to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params:,}')


### 9. Loss: WeightedFocalLoss

Focal loss + label smoothing dengan bobot inverse-frequency.
Guard: bobot minoritas > mayoritas.


In [ ]:
# ============================================================
#  WEIGHTED FOCAL LOSS — inverse-frequency + gamma + label smoothing
# ============================================================
class WeightedFocalLoss(nn.Module):
    """Focal loss + label smoothing dengan bobot inverse-frequency.
    
    Guard: bobot minoritas harus > mayoritas.
    """
    def __init__(self, num_classes=2, class_counts=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        if class_counts is None or len(class_counts) != num_classes or min(class_counts) <= 0:
            raise ValueError(f'class_counts wajib {num_classes} nilai positif, dapat {class_counts}')
        self.num_classes = num_classes
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        counts = torch.tensor(list(class_counts), dtype=torch.float32)
        cw = counts.sum() / (num_classes * counts)      # inverse-frequency N/(2*n_i)
        cw = cw / cw.mean()                             # rata-rata 1, rasio tetap
        if cw[counts.argmin()] <= cw[counts.argmax()]:
            raise ValueError(f'Bobot kelas minoritas harus > mayoritas: {cw.tolist()}')
        self.register_buffer('class_weights', cw)
        print(f'[LOSS] class_weights={cw.tolist()} (inverse-frequency, minoritas > mayoritas)')
    
    def forward(self, logits, targets):
        """Forward pass: focal loss + label smoothing."""
        # Label smoothing
        smooth_targets = torch.full_like(logits, self.label_smoothing / (self.num_classes - 1))
        smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
        
        # Focal loss
        # TODO
        
        # Cross entropy dengan class weight
        ce_loss = -log_probs * smooth_targets
        ce_loss = ce_loss.sum(dim=1)
        weighted_ce = ce_loss * self.class_weights[targets] * focal_weight
        
        return weighted_ce.mean()


def get_class_counts(df):
    """Hitung jumlah sampel per kelas dari DataFrame."""
    counts = np.bincount(df['label'].values.astype(int))
    return [int(c) for c in counts]

# Contoh penggunaan:
# cw = get_class_counts(train_sub)
# criterion = WeightedFocalLoss(num_classes=2, class_counts=cw, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTH)


### 10. Training Loop

Training dengan:
- AdamW lr 1e-3 (tanpa pretrain, tanpa freeze)
- CosineAnnealing + warmup 1 epoch
- AMP torch.amp
- Grad clip 1.0
- Monitor EMA-smooth val AUC (alpha 0.5)
- Patience 12
- SWA rata-rata 3 epoch terakhir dengan acceptance check
- Save atomic (tmp+os.replace)
- Resume dari `*_last_full.pt`
- Handler KeyboardInterrupt
- SEEDS [42, 43, 44]


In [ ]:
# ============================================================
#  TRAINING LOOP — background-safe dengan AMP, SWA, EMA
# ============================================================
def _ema_smooth(current_ema, new_value, alpha=EMA_ALPHA):
    """EMA smoothing: ema = alpha * new + (1 - alpha) * ema."""
    return alpha * new_value + (1 - alpha) * current_ema


def _average_states(states):
    """Rata-rata state_dict float antar epoch."""
    avg = OrderedDict()
    for k in states[0]:
        if states[0][k].dtype.is_floating_point:
            avg[k] = torch.stack([s[k].float() for s in states]).mean(0).to(states[0][k].dtype)
        else:
            avg[k] = states[0][k].clone()
    return avg


def save_atomic(payload, path):
    """Simpan atomic: tulis ke .tmp lalu os.replace (anti-korup)."""
    import tempfile as _tf
    _d = os.path.dirname(path) or '.'
    os.makedirs(_d, exist_ok=True)
    fd, tmp = _tf.mkstemp(dir=_d, prefix='tmp_ckpt_')
    os.close(fd)
    try:
        torch.save(payload, tmp)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            try:
                os.remove(tmp)
            except OSError:
                print(f'[save_atomic] warning: could not remove tmp file')


def train_3d(model, train_loader, val_loader, epochs=EPOCHS, lr=LR,
             class_counts=None, seed=42, label='model'):
    """Training 3D cardio classification.
    
    Args:
        model, train_loader, val_loader
        epochs, lr, class_counts
        seed: random seed
        label: identifier untuk checkpoint naming
    Returns:
        model, history
    """
    set_seed(seed)
    
    # Loss
    cw = get_class_counts(train_loader.dataset.df) if class_counts is None else class_counts
    criterion = WeightedFocalLoss(num_classes=2, class_counts=cw,
                                   gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTH)
    criterion = criterion.to(device)
    
    # Optimizer: AdamW lr 1e-3 (tanpa pretrain, tanpa freeze)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    
    # Scheduler: warmup + cosine
    if WARMUP_EPOCHS > 0 and WARMUP_EPOCHS < epochs:
        warmup = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
        cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - WARMUP_EPOCHS)
        scheduler = optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine],
                                                    milestones=[WARMUP_EPOCHS])
    else:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    # AMP
    use_amp = AMP and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    
    # Paths
    best_path = os.path.join(OUTPUT_DIR, f'{label}_best.pt')
    last_path = os.path.join(OUTPUT_DIR, f'{label}_last_full.pt')
    start_epoch = 1
    best_score, best_epoch, bad_epochs = -1.0, 0, 0
    history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_auc_smooth': []}
    val_auc_ema = None
    swa_states = []
    
    # Resume
    if os.path.exists(last_path):
        try:
            ck = torch.load(last_path, map_location=device, weights_only=False)
            model.load_state_dict(ck['model'])
            optimizer.load_state_dict(ck['optimizer'])
            if 'scheduler' in ck and ck['scheduler'] is not None:
                try: scheduler.load_state_dict(ck['scheduler'])
                except Exception as e: print(f'[{label}] scheduler load warning: {e}')
            if 'scaler' in ck and ck['scaler'] is not None and use_amp:
                try: scaler.load_state_dict(ck['scaler'])
                except Exception as e: print(f'[{label}] scaler load warning: {e}')
            best_score = ck.get('best_score', best_score)
            best_epoch = ck.get('best_epoch', 0)
            bad_epochs = ck.get('bad_epochs', 0)
            history = ck.get('history', history)
            start_epoch = int(ck.get('epoch', 0)) + 1
            val_auc_ema = ck.get('val_auc_ema', None)
            print(f'[{label}] ▶ resume dari epoch {start_epoch} (best_auc={best_score:.4f} @epoch {best_epoch})')
        except Exception as e:
            print(f'[{label}] peringatan: gagal resume ({e}) → mulai dari awal.')
    
    try:
        for epoch in range(start_epoch, epochs + 1):
            model.train()
            tot_loss, tot = 0.0, 0
            for xb, yb, _, _ in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda', enabled=use_amp):
                    logits = model(xb)
                    loss = criterion(logits, yb)
                scaler.scale(loss).backward()
                if GRAD_CLIP:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                tot_loss += loss.item() * len(yb)
                tot += len(yb)
            scheduler.step()
            
            # Evaluasi val
            model.eval()
            val_probs, val_labels = [], []
            with torch.no_grad():
                for xb, yb, _, _ in val_loader:
                    p = torch.softmax(model(xb.to(device)), dim=1)[:, 1]
                    val_probs.extend(p.cpu().tolist())
                    val_labels.extend(yb.tolist())
            val_auc = float(roc_auc_score(val_labels, val_probs)) if len(set(val_labels)) > 1 else 0.0
            tl = tot_loss / max(tot, 1)
            
            # EMA-smooth val AUC
            if val_auc_ema is None:
                val_auc_ema = val_auc
            else:
                val_auc_ema = _ema_smooth(val_auc_ema, val_auc, alpha=EMA_ALPHA)
            
            history['train_loss'].append(tl)
            history['val_auc'].append(val_auc)
            history['val_auc_smooth'].append(val_auc_ema)
            
            score = val_auc_ema
            print(f'[{label}] epoch {epoch:02d}/{epochs} | train_loss={tl:.4f} | val_auc={val_auc:.4f} | val_auc_smooth={val_auc_ema:.4f}', flush=True)
            
            if score > best_score:
                best_score, best_epoch, bad_epochs = score, epoch, 0
                save_atomic({'model': model.state_dict(),
                             'meta': dict(label=label, seed=seed, best_auc=best_score,
                                          best_epoch=best_epoch, config=CONFIG)},
                            best_path)
            else:
                bad_epochs += 1
            
            # Save last_full tiap epoch (atomic)
            save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                         'scheduler': scheduler.state_dict(),
                         'scaler': scaler.state_dict() if use_amp else None,
                         'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                         'bad_epochs': bad_epochs, 'history': history,
                         'val_auc_ema': val_auc_ema,
                         'meta': dict(label=label, seed=seed, config=CONFIG)},
                        last_path)
            
            # SWA: kumpulkan state epoch terakhir
            if SWA_EPOCHS > 0 and epoch >= epochs - SWA_EPOCHS + 1:
                swa_states.append({k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
                swa_states = swa_states[-SWA_EPOCHS:]
            
            if bad_epochs >= PATIENCE:
                print(f'  >> Early stop di epoch {epoch} (best_auc={best_score:.4f} @epoch {best_epoch})')
                break
    
    except KeyboardInterrupt:
        try:
            save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                         'scheduler': scheduler.state_dict(),
                         'scaler': scaler.state_dict() if use_amp else None,
                         'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                         'bad_epochs': bad_epochs, 'history': history,
                         'val_auc_ema': val_auc_ema,
                         'meta': dict(label=label, seed=seed, config=CONFIG)},
                        os.path.join(OUTPUT_DIR, f'{label}_interrupted_full.pt'))
            print(f'[{label}] ⚠ interrupted di epoch {epoch} — state tersimpan.')
        except Exception as e:
            print(f'[{label}] gagal menyimpan interrupted state: {e}')
        raise
    
    # Load best model
    if os.path.exists(best_path):
        _best = torch.load(best_path, map_location=device, weights_only=False)
        model.load_state_dict(_best['model'] if isinstance(_best, dict) and 'model' in _best else _best)
    
    # SWA acceptance check
    if SWA and len(swa_states) > 1:
        _swa_weights = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(_average_states(swa_states))
        # Eval SWA di val
        model.eval()
        swa_probs, swa_labels = [], []
        with torch.no_grad():
            for xb, yb, _, _ in val_loader:
                p = torch.softmax(model(xb.to(device)), dim=1)[:, 1]
                swa_probs.extend(p.cpu().tolist())
                swa_labels.extend(yb.tolist())
        swa_auc = float(roc_auc_score(swa_labels, swa_probs)) if len(set(swa_labels)) > 1 else 0.0
        if swa_auc >= best_score:
            print(f'[{label}] SWA dipakai: val_auc={swa_auc:.4f} >= best {best_score:.4f}')
        else:
            model.load_state_dict(_swa_weights)
            print(f'[{label}] SWA ditolak: val_auc={swa_auc:.4f} < best {best_score:.4f}')
    
    print(f'[{label}] Selesai. Best val_auc = {best_score:.4f} (epoch {best_epoch})')
    return model, history


### 11. Threshold Tuning

Scan threshold 0.05-0.95 step 0.02, clamp [0.30, 0.70], min gain 0.02 vs 0.5,
anti-degenerate (tolak recall>=0.999 atau pos_rate>0.85).


In [ ]:
# ============================================================
#  THRESHOLD TUNING — HANYA di VAL (anti-overfit)
# ============================================================
def tune_threshold(val_df, val_probs, val_labels, step=TUNE_STEP):
    """Scan threshold optimal pd data VALIDASI.
    
    - scan t ∈ [0.05, 0.95] step 0.02
    - clamp [0.30, 0.70]
    - min gain 0.02 vs 0.5
    - anti-degenerate: tolak recall>=0.999 atau pos_rate>0.85
    Return best_threshold.
    """
    val_probs = np.array(val_probs)
    val_labels = np.array(val_labels)
    
    best_t, best_f1 = 0.5, -1.0
    for t in np.arange(0.05, 0.95 + 1e-9, step):
        yp = (val_probs >= t).astype(int)
        if len(set(yp)) < 2:
            continue
        f1 = f1_score(val_labels, yp, zero_division=0)
        recall = recall_score(val_labels, yp, zero_division=0)
        pos_rate = float(yp.mean())
        
        # Anti-degenerate
        if recall >= TUNE_TOLERATE or pos_rate > TUNE_MAX_POSRATE:
            continue
        
        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)
    
    # Clamp
    best_t = float(np.clip(best_t, TUNE_CLAMP[0], TUNE_CLAMP[1]))
    
    # Min gain vs 0.5
    base_f1 = f1_score(val_labels, (val_probs >= 0.5).astype(int), zero_division=0)
    if best_f1 - base_f1 < TUNE_MIN_GAIN:
        best_t, best_f1 = 0.5, base_f1
    
    print(f'[THRESHOLD] Best t={best_t:.2f} (f1={best_f1:.4f}, base_0.5={base_f1:.4f})')
    return best_t


def get_val_probs(model, val_loader):
    """Dapatkan probabilitas dan label dari val loader."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for xb, yb, _, _ in val_loader:
            p = torch.softmax(model(xb.to(device)), dim=1)[:, 1]
            probs.extend(p.cpu().tolist())
            labels.extend(yb.tolist())
    return np.array(probs), np.array(labels)


### 12. Evaluasi TEST + Bootstrap 95% CI

Evaluasi TEST **sekali** di akhir. Metrik: acc/prec/rec/f1/auc + bootstrap 95% CI
per-subjek (n=2000). Confusion matrix. Simpan `config_snapshot.json` + `results.csv`.


In [ ]:
# ============================================================
#  EVALUASI TEST + BOOTSTRAP CI
# ============================================================
def compute_metrics(y_true, y_pred, y_prob):
    """Hitung semua metrik."""
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'auc':       roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan'),
    }


def bootstrap_ci(y_true, y_prob, thr, n_boot=BOOTSTRAP_N, seed=0):
    """Bootstrap 95% CI F1 & AUC — per-subjek."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob, dtype=float)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    f1s, aucs = [], []
    for _ in range(n_boot):
        s = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y_true[s])) < 2:
            continue
        f1s.append(f1_score(y_true[s], (y_prob[s] >= thr).astype(int), zero_division=0))
        aucs.append(roc_auc_score(y_true[s], y_prob[s]))
    f1_ci = (float(np.percentile(f1s, 2.5)), float(np.percentile(f1s, 97.5)))
    auc_ci = (float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5)))
    return f1_ci, auc_ci, len(y_true)


def evaluate_test(model, test_loader, threshold):
    """Evaluasi TEST sekali. Return metrics + records."""
    model.eval()
    records, probs, labels = [], [], []
    with torch.no_grad():
        for xb, yb, subj, _ in test_loader:
            p = torch.softmax(model(xb.to(device)), dim=1)[:, 1]
            probs.extend(p.cpu().tolist())
            labels.extend(yb.tolist())
            records.extend([(s, int(yt), float(pr)) for s, yt, pr in zip(subj, yb.tolist(), p.tolist())])
    
    y_pred = [1 if p >= threshold else 0 for p in probs]
    metrics = compute_metrics(labels, y_pred, probs)
    
    # Confusion matrix
    cm = confusion_matrix(labels, y_pred)
    
    # Bootstrap CI per-subjek
    subj_probs = {}
    subj_labels = {}
    for s, yt, pr in records:
        if s not in subj_probs:
            subj_probs[s] = []
            subj_labels[s] = []
        subj_probs[s].append(pr)
        subj_labels[s].append(yt)
    
    subj_prob_arr = np.array([np.mean(subj_probs[s]) for s in sorted(subj_probs.keys())])
    subj_label_arr = np.array([subj_labels[s][0] for s in sorted(subj_probs.keys())])
    
    f1_ci, auc_ci, n_subj = bootstrap_ci(subj_label_arr, subj_prob_arr, threshold)
    
    return metrics, cm, f1_ci, auc_ci, n_subj, records


def run_evaluation(model, test_loader, threshold):
    """Jalankan evaluasi TEST lengkap."""
    metrics, cm, f1_ci, auc_ci, n_subj, records = evaluate_test(model, test_loader, threshold)
    
    print('\n===== EVALUASI TEST =====')
    print(f'  Accuracy:  {metrics["accuracy"]:.4f}')
    print(f'  Precision: {metrics["precision"]:.4f}')
    print(f'  Recall:    {metrics["recall"]:.4f}')
    print(f'  F1:        {metrics["f1"]:.4f}')
    print(f'  AUC:       {metrics["auc"]:.4f}')
    print(f'  N subjects: {n_subj}')
    print(f'\n  Confusion Matrix:')
    print(f'    TN={cm[0,0]}  FP={cm[0,1]}')
    print(f'    FN={cm[1,0]}  TP={cm[1,1]}')
    print(f'\n  Bootstrap 95% CI (per-subjek, n={n_subj}):')
    print(f'    F1: [{f1_ci[0]:.4f}, {f1_ci[1]:.4f}]')
    print(f'    AUC: [{auc_ci[0]:.4f}, {auc_ci[1]:.4f}]')
    
    return metrics, cm, f1_ci, auc_ci, n_subj, records


### 13. Pipeline Eksekusi

Jalankan seluruh pipeline: load data → train (per seed) → threshold tuning → eval TEST → simpan.


In [ ]:
# ============================================================
#  PIPELINE — Eksekusi lengkap
# ============================================================
def run_pipeline():
    """Jalankan pipeline lengkap: train → tune → eval → save."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Buat loaders
    train_ld, val_ld, test_ld = make_3d_loaders(train_sub, val_sub, test_df)
    print(f'Loaders: train={len(train_ld.dataset)} val={len(val_ld.dataset)} test={len(test_ld.dataset)}')
    
    all_results = {}
    all_thresholds = {}
    
    for seed in SEEDS:
        print(f'\n{"="*60}')
        print(f'SEED {seed}')
        print(f'{"="*60}')
        set_seed(seed)
        
        # Build model
        model = build_model().to(device)
        
        # Train
        model, history = train_3d(model, train_ld, val_ld, epochs=EPOCHS, lr=LR,
                                   class_counts=get_class_counts(train_sub),
                                   seed=seed, label=f'cardio_seed{seed}')
        
        # Threshold tuning di VAL
        val_probs, val_labels = get_val_probs(model, val_ld)
        threshold = tune_threshold(val_sub, val_probs, val_labels)
        all_thresholds[seed] = threshold
        
        # Evaluasi TEST
        test_metrics, test_cm, f1_ci, auc_ci, n_subj, records = evaluate_test(model, test_ld, threshold)
        all_results[seed] = {
            'metrics': test_metrics,
            'confusion_matrix': test_cm.tolist(),
            'f1_ci': f1_ci,
            'auc_ci': auc_ci,
            'n_subj': n_subj,
            'threshold': threshold,
        }
        print(f'  TEST: acc={test_metrics["accuracy"]:.4f} f1={test_metrics["f1"]:.4f} auc={test_metrics["auc"]:.4f}')
    
    # Best seed berdasarkan AUC
    best_seed = max(all_results, key=lambda s: all_results[s]['metrics']['auc'])
    print(f'\nBest seed: {best_seed} (AUC={all_results[best_seed]["metrics"]["auc"]:.4f})')
    
    return all_results, all_thresholds, best_seed

# Jalankan pipeline
results, thresholds, best_seed = run_pipeline()


### 14. Evaluasi Final

Simpan `config_snapshot.json`, `results.csv`, `confusion_matrix.csv`, dan zip outputs ke `ncct_3d_cardio_weights.zip`.


In [ ]:
# Config snapshot
with open(os.path.join(OUTPUT_DIR, 'config_snapshot.json'), 'w') as f:
    json.dump(CONFIG, f, indent=2, default=str)
print('✔ config_snapshot.json tersimpan')

# Results CSV
results_rows = []
for seed in SEEDS:
    r = results[seed]
    row = {
        'seed': seed,
        'threshold': r['threshold'],
        'accuracy': r['metrics']['accuracy'],
        'precision': r['metrics']['precision'],
        'recall': r['metrics']['recall'],
        'f1': r['metrics']['f1'],
        'auc': r['metrics']['auc'],
        'f1_ci_lo': r['f1_ci'][0],
        'f1_ci_hi': r['f1_ci'][1],
        'auc_ci_lo': r['auc_ci'][0],
        'auc_ci_hi': r['auc_ci'][1],
        'n_subj': r['n_subj'],
    }
    results_rows.append(row)

results_df = pd.DataFrame(results_rows)
results_df.to_csv(os.path.join(OUTPUT_DIR, 'results.csv'), index=False)
print('✔ results.csv tersimpan')

# Confusion matrix (best seed)
best_cm = np.array(results[best_seed]['confusion_matrix'])
pd.DataFrame(best_cm, columns=['Pred_NonCardio', 'Pred_Cardio'],
             index=['Actual_NonCardio', 'Actual_Cardio']).to_csv(
    os.path.join(OUTPUT_DIR, 'confusion_matrix.csv'))
print('✔ confusion_matrix.csv tersimpan')

# Summary
print(f'\n===== RINGKASAN FINAL =====')
print(f'Best seed: {best_seed}')
print(f'Best AUC: {results[best_seed]["metrics"]["auc"]:.4f}')
print(f'Best F1: {results[best_seed]["metrics"]["f1"]:.4f}')
print(f'Bootstrap F1 CI: [{results[best_seed]["f1_ci"][0]:.4f}, {results[best_seed]["f1_ci"][1]:.4f}]')
print(f'Bootstrap AUC CI: [{results[best_seed]["auc_ci"][0]:.4f}, {results[best_seed]["auc_ci"][1]:.4f}]')

# Zip outputs (Kaggle: /kaggle/working, Colab: /content — keduanya bisa diunduh dari UI)
zip_path = os.path.join(OUTPUT_DIR, 'ncct_3d_cardio_weights.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in glob.glob(os.path.join(OUTPUT_DIR, '*.pt')) + \
              glob.glob(os.path.join(OUTPUT_DIR, '*.json')) + \
              glob.glob(os.path.join(OUTPUT_DIR, '*.csv')):
        zf.write(fp, arcname=os.path.basename(fp))
print(f'✔ {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')

print('\n✅ Pipeline selesai!')
